# Lang2Vec Signal Overview

Notebook to inspect Lang2Vec coverage for FineWeb languages, the available feature sets, and how different distance types (and weighted combinations) behave."


In [1]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path("/home/ubuntu/dice_repos/HTYLLM-PG/approaches/CoLA")
FW_LANG_PATH = ROOT / 'data_prep' / 'base_data' / 'fineweb2-language-distribution.csv'
LANG2VEC_DIR = ROOT / 'data_prep' / 'lang_cluster_analysis' / 'lang2vec'
sys.path.insert(0, str(LANG2VEC_DIR))
from lang2vec import lang2vec as l2v

fw_df = pd.read_csv(FW_LANG_PATH, usecols=['code', 'family', 'resource_availability']).drop_duplicates('code')
all_distance_langs = set(getattr(l2v, 'DISTANCE_LANGUAGES', l2v.available_distance_languages()))
overlap_df = fw_df[fw_df['code'].isin(all_distance_langs)].reset_index(drop=True)
overlap_langs = overlap_df['code'].tolist()
print(f"Overlap languages: {len(overlap_langs)} across {overlap_df['family'].nunique()} families")


Overlap languages: 1804 across 102 families


In [2]:

dist_types = list(l2v.DISTANCES)
stats = []
sample_langs = overlap_langs[:200]
for dist in dist_types:
    matrix = np.asarray(l2v.distance(dist, sample_langs), dtype=float)
    tri = np.triu_indices_from(matrix, k=1)
    vals = matrix[tri]
    stats.append(
        {
            'distance': dist,
            'min': float(vals.min()),
            'median': float(np.median(vals)),
            'max': float(vals.max()),
            'mean': float(vals.mean()),
        }
    )
pd.DataFrame(stats)


,distance,min,median,max,mean
0,genetic,0.0556,1.0000,1.0000,0.970787
1,geographic,0.0000,0.5000,1.0000,0.468357
2,syntactic,0.0000,0.7300,1.0000,0.653256
3,inventory,0.0000,0.6030,0.7566,0.450940
4,phonological,0.0000,0.0002,1.0000,0.324568
5,featural,0.0000,0.7000,1.0000,0.708673


In [7]:

feature_sets = sorted(l2v.available_feature_sets())
rows = []
for feature in feature_sets:
    try:
        feats = l2v.get_features(overlap_langs, feature)
        count = sum(1 for lang in overlap_langs if lang in feats and all(val != '--' for val in feats[lang]))
    except Exception as exc:
        count = f"error: {exc}"
    rows.append({'feature_set': feature, 'complete_langs': count})
pd.DataFrame(rows)


,feature_set,complete_langs
0,fam,1804
1,geo,1804
2,id,1804
3,inventory_average,756
4,inventory_ethnologue,0
5,inventory_knn,1804
6,inventory_phoible_aa,132
7,inventory_phoible_gm,158
8,inventory_phoible_ph,168
9,inventory_phoible_ra,41


### joel notes
- Genetic: Language family relationships.
- Syntactic: Grammatical and word-order properties.
- Phonological: Sound-system patterns.
- Inventory: Which sounds the language has.

but we clearly see that most features dont have enough data, however we als see that the knn features have enough data. SO what are those?

### knn features
the features syntax_knn, phonology_knn, inventory_knn have data for all our langs, but why?
- They predicted missing tyological feature values  using knn 
- they use weighted 10-nearest neigbors classifcaition
- the distance used for finding neigbors is an average or genetic, geographic, and featureal distances between the langs
- here is teh paper for reference https://aclanthology.org/E17-2002.pdf
- this is what they say spcifically "By taking an average of genetic, geographical, and feature distances between
languages, and calculating a weighted 10-nearestneighbors classification, we can predict feature
missing values with an accuracy of 92.93% in 10-
fold cross-validation."

In [8]:

def inspect_pairs(distance_type, n=5):
    matrix = np.asarray(l2v.distance(distance_type, overlap_langs[:200]), dtype=float)
    tri = np.triu_indices_from(matrix, k=1)
    vals = matrix[tri]
    order = np.argsort(vals)
    pairs = []
    for idx in order[:n]:
        i, j = tri[0][idx], tri[1][idx]
        pairs.append((overlap_langs[i], overlap_langs[j], float(vals[idx])))
    far = []
    for idx in order[-n:]:
        i, j = tri[0][idx], tri[1][idx]
        far.append((overlap_langs[i], overlap_langs[j], float(vals[idx])))
    return pairs, far

close_pairs, far_pairs = inspect_pairs('syntactic', n=5)
print('Closest syntactic pairs:')
for a, b, d in close_pairs:
    print(f"  {a} – {b}: {d:.3f}")
print('\nMost distant syntactic pairs:')
for a, b, d in far_pairs:
    print(f"  {a} – {b}: {d:.3f}")


Closest syntactic pairs:
  atd – bgt: 0.000
  abx – atg: 0.000
  aha – anv: 0.000
  aln – bba: 0.000
  aln – bgz: 0.000

Most distant syntactic pairs:
  atb – bbo: 1.000
  bjr – blw: 1.000
  aso – bis: 1.000
  aso – bib: 1.000
  aso – blw: 1.000


In [9]:

weights = {'genetic': 0.4, 'syntactic': 0.3, 'phonological': 0.2, 'inventory': 0.1}

sample = overlap_langs[:80]
n = len(sample)
combined = np.zeros((n, n), dtype=float)

for dist, w in weights.items():
    combined += w * np.asarray(l2v.distance(dist, sample), dtype=float)

np.fill_diagonal(combined, 0.0)

tri = np.triu_indices_from(combined, k=1)
vals = combined[tri]

print(
    f"Weighted combo stats -> "
    f"min {vals.min():.3f}, median {np.median(vals):.3f}, "
    f"max {vals.max():.3f}, mean {vals.mean():.3f}"
)

order = np.argsort(vals)

print("\nTop 5 closest pairs (weighted):")
for idx in order[:5]:
    i, j = tri[0][idx], tri[1][idx]
    print(f"  {sample[i]} – {sample[j]}: {vals[idx]:.3f}")

print("\nTop 5 farthest pairs (weighted):")
for idx in order[-5:]:
    i, j = tri[0][idx], tri[1][idx]
    print(f"  {sample[i]} – {sample[j]}: {vals[idx]:.3f}")



Weighted combo stats -> min 0.114, median 0.698, max 0.951, mean 0.694

Top 5 closest pairs (weighted):
  adh – alz: 0.114
  akb – alj: 0.160
  agw – alj: 0.160
  aaz – alj: 0.160
  abx – alj: 0.160

Top 5 farthest pairs (weighted):
  adl – aeb: 0.933
  aeb – ahk: 0.941
  ach – aeb: 0.943
  aeb – amn: 0.947
  aeb – agr: 0.951


In [11]:
sample_langs = ['eng', 'spa', 'tam', 'zho', 'amh']

syntax_knn = l2v.get_features(sample_langs, 'syntax_knn', header=True)
print('syntax_knn feature names (first 10):', syntax_knn['CODE'][:10])

for lang in sample_langs:
    print(f"{lang}: {syntax_knn[lang][:10]}")


syntax_knn feature names (first 10): [np.str_('S_SVO'), np.str_('S_SOV'), np.str_('S_VSO'), np.str_('S_VOS'), np.str_('S_OVS'), np.str_('S_OSV'), np.str_('S_SUBJECT_BEFORE_VERB'), np.str_('S_SUBJECT_AFTER_VERB'), np.str_('S_OBJECT_AFTER_VERB'), np.str_('S_OBJECT_BEFORE_VERB')]
eng: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(1.0), np.float64(0.0)]
spa: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(0.0)]
tam: [np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(1.0)]
zho: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(1.0), np.float64(0.0)]
amh: [np.float64(0

In [12]:

sample_langs = ['eng', 'spa', 'tam', 'zho', 'amh']
for feature in ['syntax_knn', 'phonology_knn', 'inventory_knn']:
    feats = l2v.get_features(sample_langs, feature, header=True)
    print(f"Feature set: {feature}")
    print('Feature names (first 10):', feats['CODE'][:10])
    for lang in sample_langs:
        print(f"  {lang}: {feats[lang][:10]}")
    print()


Feature set: syntax_knn
Feature names (first 10): [np.str_('S_SVO'), np.str_('S_SOV'), np.str_('S_VSO'), np.str_('S_VOS'), np.str_('S_OVS'), np.str_('S_OSV'), np.str_('S_SUBJECT_BEFORE_VERB'), np.str_('S_SUBJECT_AFTER_VERB'), np.str_('S_OBJECT_AFTER_VERB'), np.str_('S_OBJECT_BEFORE_VERB')]
  eng: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(1.0), np.float64(0.0)]
  spa: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(0.0)]
  tam: [np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(1.0)]
  zho: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(1.0), np.float64(0.0

In [13]:

sample_langs = ['eng', 'spa', 'tam', 'zho', 'amh']
knn_sets = {
    'syntax_knn': l2v.get_features(sample_langs, 'syntax_knn', header=True),
    'phonology_knn': l2v.get_features(sample_langs, 'phonology_knn', header=True),
    'inventory_knn': l2v.get_features(sample_langs, 'inventory_knn', header=True),
}

rows = []
for feature, feats in knn_sets.items():
    codes = feats['CODE']
    n_features = len(codes)
    variability = []
    for col in range(n_features):
        values = [feats[lang][col] for lang in sample_langs]
        if isinstance(values[0], str):
            continue
        if max(values) - min(values) > 0:
            variability.append(col)
    rows.append(
        {
            'feature_set': feature,
            'dimensions': n_features,
            'non_constant_dims_in_sample': len(variability),
        }
    )

pd.DataFrame(rows)


,feature_set,dimensions,non_constant_dims_in_sample
0,syntax_knn,103,55
1,phonology_knn,28,10
2,inventory_knn,158,64
